In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os

# System definitions

In [ ]:
from nsflows.systems.gaussians import normal
from nsflows.systems.lennard_jones import lennard_jones
from nsflows.systems.uniforms import box_uniform

n_particles = 8
dimensions = 2
box_length = 2.9
cutin = 0.8
rho = n_particles/(box_length)**(dimensions)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

box_uniform_2D = box_uniform(n_particles=n_particles, dimensions=dimensions, device=device, box_length=box_length)
LJ_disks = lennard_jones(n_particles=n_particles, dimensions=dimensions, rho=rho, device=device, cutin=cutin, lrc=True)

## Define Parameters

In [ ]:
train = True
load_best = True

# Dataset Parameters
# DO NOT TOUCH THESE PARAMETERS UNLESS YOU KNOW WHAT YOU ARE DOING
K = 10000
data_dir = f"../data/lj/K{K}/L2.9/live_sets"
first_live_set = 1000
live_set_interval = 500

assert os.path.exists(os.path.join(data_dir, f"samples_{first_live_set}.pt")), "First live set not found"
assert os.path.exists(os.path.join(data_dir, f"samples_{first_live_set + live_set_interval}.pt")), "Live set interval is probably wrong"

live_sets = [50000, 150000, 250000, 350000, 450000]

# Conditioning Parameters
n_live_sets = 3
live_set_step = 11000
reduce_dataset = False
split_equal = False

# Training Parameters
# z -> x
w_zx = 0
k = 1.0e-3
j = 0.0

# x -> z
w_xz = 1

# Generation Parameters
n_replicas = 10
n_samples = 10000

# Output Folder Definition

In [ ]:
from nsflows.tools.util import generate_unique_identifier, remove_empty_directories, generate_output_directory

root_folder = "./output/L2.9/"
remove_empty_directories(root_folder=root_folder)

if train:
    run_id = generate_unique_identifier()
    output_dir = generate_output_directory(run_id, root_folder=root_folder)
else:
    # Re-evaluate a completed run instead of training. The trained networks are
    # ~88 MB each and are not shipped, so this needs a run you produced yourself;
    # the results of ours are in ../data/lj/K10000/L2.9/runs/xz_multiple_conditioning_window_10K
    # as the three .txt files that Figure 4 reads.
    output_dir = "./output/L2.9/<run-id>"
    print(f"Run Folder: {output_dir}")

## Loop Over Live Sets

In [ ]:
import math
import copy

from nsflows.network.flow_assembler import flow_assembler
from nsflows.network.coupling_blocks import EquivariantRQS
from nsflows.transformations.normalization import NormalizeBox
from nsflows.transformations.remove_origin import remove_origin
from nsflows.network.dataset import PBCDataset
from nsflows.network.trainer import Trainer
from nsflows.tools.util import ress

mean_ident = []
mean_gener = []
mean_RESS  = []

stderr_ident = []
stderr_gener = []
stderr_RESS  = []

for ls_id, live_set in enumerate(live_sets):

    print()
    if train:
        live_set_output_dir = generate_output_directory(f"{live_set}", root_folder=output_dir)
    else:
        live_set_output_dir = os.path.join(output_dir, f"{live_set}")
        print(f"Live Set Folder: {live_set_output_dir}")
        
    # Network Definition
    conditioned = True
    n_blocks = 28
    n_bins = 12

    block_unit = [
        
        EquivariantRQS((0,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
        EquivariantRQS((1,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
        
        EquivariantRQS((1,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
        EquivariantRQS((0,), n_particles-1, dimensions, device, left=-1, bottom=-1, right=1, top=1, n_bins=n_bins, conditioned=conditioned),
    ]
    block_list = [copy.deepcopy(element) for block in range(n_blocks) for element in block_unit]

    box_pr = torch.from_numpy(np.array([box_length, box_length], dtype=np.float32)).to(device)
    box_sys = torch.from_numpy(np.array([box_length, box_length], dtype=np.float32)).to(device)

    norm_box_pr = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_pr, device=device)
    norm_box_sys = NormalizeBox(n_particles=n_particles, dimensions=dimensions, box_length=box_sys, device=device)
    rm_origin = remove_origin(n_particles=n_particles, dimensions=dimensions, device=device)

    # Flow Initialization  
    flow = flow_assembler(box_uniform_2D, LJ_disks, device=device, 
                        blocks = block_list,
                        prior_sided_transformation_layers = [norm_box_pr, rm_origin], 
                        post_sided_transformation_layers = [norm_box_sys, rm_origin],
                        k=k, j=j,
                        ).to(device)

    flow_parameters = sum(p.numel() for p in flow.parameters() if p.requires_grad)
    print(f"Network parameters: {flow_parameters}")

    # Load Data    
    data_samples = []
    data_conditions = []
    
    print("Training with data from: ")
    for i in range(n_live_sets):
        
        # This handles the first case in which the reference is less than n_live_set*live_set_step (e.g. 5000)
        if live_set < n_live_sets*live_set_step:
            multiples = [n for n in range(first_live_set, live_set + 1) if n % live_set_interval == 0]
            if len(multiples) >= n_live_sets:
                chosen = list(np.linspace(multiples[0], multiples[-1], n_live_sets, dtype=int))
                chosen = [((n + live_set_interval//2) // live_set_interval) * live_set_interval for n in chosen]
                dataset_index = chosen[i]
            else:
                raise Exception(f"Cannot sample {n_live_sets} elements from a set of {len(multiples)} elements w/o replacement")        
        else:
            dataset_index = live_set -i*live_set_step
        data_samples.append(torch.load(os.path.join(data_dir, f"samples_{dataset_index}.pt")))
        data_conditions.append(torch.load(os.path.join(data_dir, f"U_max_{dataset_index}.pt")).repeat(data_samples[0].shape[0]))

        print(f"- dataset_index: {dataset_index} - Umax = {data_conditions[i][0].item()}")

    data_samples = torch.cat(data_samples, dim=0)
    data_conditions = torch.cat(data_conditions, dim=0)

    if reduce_dataset:
        if split_equal:
            K_red = K//n_live_sets
            indx = np.random.choice(np.arange(K_red), K_red, replace=False)
            for set in range(1, n_live_sets):
                indx = np.concatenate([indx, np.random.choice(np.arange(set*K_red, (set+1)*K_red), K_red, replace=False)])
            if indx.shape[0] < K:
                indx = np.concatenate([indx, np.random.choice(np.arange(K), K - indx.shape[0], replace=False)])
            indx = np.random.choice(np.arange(n_live_sets*K), K, replace=False)
        data_samples = data_samples[indx]
        data_conditions = data_conditions[indx]

    # Initialize Dataset
    samples_dataset = PBCDataset(flow=flow, 
                                        data_tensor=data_samples, 
                                        test_fraction=0.1, 
                                        shuffle_data=True, 
                                        conditions_tensor=data_conditions,
                                        transform=True,
                                        augment=True)
    
    # Training/Load Parameters
    if train:

        # Initialize Trainer
        flow_trainer = Trainer(flow)
        
        # Train Flow
        batch_size = 750
        steps_per_epoch = math.ceil(len(samples_dataset) / batch_size)
        total_steps = 4500
        # n_epochs = 500
        # total_steps = n_epochs*steps_per_epoch
        n_epochs = math.ceil(total_steps / steps_per_epoch)
        exact_epochs = total_steps / steps_per_epoch

        optimizer = torch.optim.Adam([p for p in flow.parameters() if p.requires_grad], lr=1e-4)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=2.5e-4,                      # peak LR
            epochs=n_epochs,
            steps_per_epoch=steps_per_epoch,
        )
        # scheduler = None

        print()
        print(f"Training:")
        metrics = flow_trainer.training_routine(samples_dataset, 
                            w_zx=w_zx, 
                            w_xz=w_xz, 
                            n_epochs=n_epochs, 
                            batch_size=batch_size, 
                            n_dump=1, 
                            n_save=0,
                            save_best=True,
                            save_dir=live_set_output_dir,
                            optimizer=optimizer, 
                            scheduler=scheduler,
                            clip_grad_norm=100, 
        )

        if load_best:
            flow_parameters_filepath = os.path.join(live_set_output_dir, f"best_flow_parameters.pt")
            print(f"Loading network parameters from {flow_parameters_filepath}")
            flow.load_state_dict(torch.load(flow_parameters_filepath))

        length_x = 60
        fig_size = (length_x * 0.393701, 10 * 0.393701)
        fig, ax = plt.subplots(1, 5, figsize = fig_size, dpi = 400, tight_layout=True)

        ax[0].plot(metrics[:,0], metrics[:,2], label="train")
        ax[0].plot(metrics[:,0], metrics[:,6], label="eval")
        ax[0].set_xlabel("epochs")
        ax[0].set_ylabel("NLL loss")

        ax[1].plot(metrics[:,0], metrics[:,3], label="train")
        ax[1].plot(metrics[:,0], metrics[:,7], label="eval")
        ax[1].set_xlabel("epochs")
        ax[1].set_ylabel("ECUT loss")

        ax[2].plot(metrics[:,0], metrics[:,4], label="train")
        ax[2].plot(metrics[:,0], metrics[:,8], label="eval")
        ax[2].set_xlabel("epochs")
        ax[2].set_ylabel("ECUT violation")
        ax[2].legend(frameon=False)
        # ax[2].set_yscale("log")

        ax[3].plot(metrics[:,0], metrics[:,5], color="C0", label="train")
        ax[3].set_xlabel("epochs")
        ax[3].set_ylabel("Gradient Norm")
        # ax[3].set_ylim(0, 1)
        ax[3].legend(frameon=False)
        ax[3].set_yscale("log")

        ax[4].plot(metrics[:,0], metrics[:,9], color="C1", label="eval")
        ax[4].set_xlabel("epochs")
        ax[4].set_ylabel("RESS")
        ax[4].set_ylim(0, 1)
        ax[4].legend(frameon=False)

        # plt.savefig(os.path.join(live_set_output_dir, f"metrics.png"))
        plt.show()

    else:

        if load_best:
            flow_parameters_filepath = os.path.join(live_set_output_dir, f"best_flow_parameters.pt")
        else:
            flow_parameters_filepath = os.path.join(live_set_output_dir, f"flow_parameters.pt")
        print(f"Loading network parameters from {flow_parameters_filepath}")
        flow.load_state_dict(torch.load(flow_parameters_filepath))
        metrics_filepath = os.path.join(live_set_output_dir, f"train_log.txt")
        print(f"Loading metrics from {metrics_filepath}")
        metrics = np.loadtxt(metrics_filepath)

        length_x = 60
        fig_size = (length_x * 0.393701, 10 * 0.393701)
        fig, ax = plt.subplots(1, 5, figsize = fig_size, dpi = 400, tight_layout=True)

        ax[0].plot(metrics[:,0], metrics[:,2], label="train")
        ax[0].plot(metrics[:,0], metrics[:,6], label="eval")
        ax[0].set_xlabel("epochs")
        ax[0].set_ylabel("NLL loss")

        ax[1].plot(metrics[:,0], metrics[:,3], label="train")
        ax[1].plot(metrics[:,0], metrics[:,7], label="eval")
        ax[1].set_xlabel("epochs")
        ax[1].set_ylabel("ECUT loss")

        ax[2].plot(metrics[:,0], metrics[:,4], label="train")
        ax[2].plot(metrics[:,0], metrics[:,8], label="eval")
        ax[2].set_xlabel("epochs")
        ax[2].set_ylabel("ECUT violation")
        ax[2].legend(frameon=False)
        # ax[2].set_yscale("log")

        ax[3].plot(metrics[:,0], metrics[:,5], color="C0", label="train")
        ax[3].set_xlabel("epochs")
        ax[3].set_ylabel("Gradient Norm")
        # ax[3].set_ylim(0, 1)
        ax[3].legend(frameon=False)
        ax[3].set_yscale("log")

        ax[4].plot(metrics[:,0], metrics[:,9], color="C1", label="eval")
        ax[4].set_xlabel("epochs")
        ax[4].set_ylabel("RESS")
        ax[4].set_ylim(0, 1)
        ax[4].legend(frameon=False)

        # plt.savefig(os.path.join(live_set_output_dir, f"metrics.png"))
        plt.show()

    # Generating Configurations below Umax
    Umax = data_conditions[0].item()

    ident = np.zeros(n_replicas)
    gener = np.zeros(n_replicas)
    RESS = np.zeros(n_replicas)

    with torch.no_grad():

        for rep in range(n_replicas):

            z = flow.prior.sample(n_samples, transform=True)
            condition_z = torch.zeros((z.shape[0], 1), device=device)

            # Transforming through normalizing flow
            target_x, logJ_zx = flow.F_zx(z, condition_z)

            energy_x_target = flow.posterior.energy(target_x)
            energy_x_identity_np = flow.posterior.energy(z).cpu().numpy()
            energy_x_target_np = energy_x_target.cpu().numpy()

            ident[rep] = np.sum(energy_x_identity_np < Umax)/n_samples
            gener[rep] = np.sum(energy_x_target_np < Umax)/n_samples

            log_prob_zx = -torch.where(energy_x_target < Umax, torch.zeros(1, device=device), torch.inf*torch.ones(1, device=device))
            log_prob_z = -flow.prior.energy(z)        
            log_w = (log_prob_zx - log_prob_z + logJ_zx).squeeze(-1)
            RESS[rep] = ress(log_w)

    mean_ident.append(ident.mean())
    mean_gener.append(gener.mean())
    mean_RESS.append(RESS.mean())

    stderr_ident.append(np.std(ident, ddof=1) / np.sqrt(n_replicas))
    stderr_gener.append(np.std(gener, ddof=1) / np.sqrt(n_replicas))
    stderr_RESS.append(np.std(RESS, ddof=1) / np.sqrt(n_replicas))

    print()
    print(f"Live Set {ls_id+1}/{len(live_sets)} RECAP - Umax = {Umax}")
    print(f"Check condition: cond_z -> Umax {samples_dataset.normalize_conditions(condition_z[0], inverse=True).item()}")
    print(f"Average Identity Efficiency: {mean_ident[ls_id]} +- {stderr_ident[ls_id]}")
    print(f"Average Generation Efficiency: {mean_gener[ls_id]} +- {stderr_gener[ls_id]}")
    print(f"Average RESS: {mean_RESS[ls_id]} +- {stderr_RESS[ls_id]}")

In [ ]:
mean_ident = np.array(mean_ident)
mean_gener = np.array(mean_gener)
mean_RESS = np.array(mean_RESS)

stderr_ident = np.array(stderr_ident)
stderr_gener = np.array(stderr_gener)
stderr_RESS = np.array(stderr_RESS)

np.savetxt(os.path.join(output_dir, "eff_identity.txt"), np.c_[np.arange(len(live_sets)), np.array(live_sets), mean_ident, stderr_ident])
np.savetxt(os.path.join(output_dir, "eff_generation.txt"), np.c_[np.arange(len(live_sets)), np.array(live_sets), mean_gener, stderr_gener])
np.savetxt(os.path.join(output_dir, "RESS.txt"), np.c_[np.arange(len(live_sets)), np.array(live_sets), mean_RESS, stderr_RESS])

In [ ]:
fig_size = (7.5 * 0.393701, 5 * 0.393701)
fig, ax = plt.subplots(figsize = fig_size, dpi = 300)

ax.errorbar(np.arange(len(live_sets)), mean_ident, yerr=stderr_ident*np.sqrt(n_replicas), capsize=2.5, color="C1", label="Identity")
ax.errorbar(np.arange(len(live_sets)), mean_gener, yerr=stderr_gener*np.sqrt(n_replicas), capsize=2.5, color="C2", label="Generated")
ax.errorbar(np.arange(len(live_sets)), mean_RESS, yerr=stderr_RESS*np.sqrt(n_replicas), capsize=2.5, color="C3", label="RESS")
ax.set_yscale("log")
ax.set_ylim(0.0001, 1)
ax.set_xlabel("Live Set")
ax.set_ylabel("Efficiency")

plt.legend(frameon=False)
plt.show()

In [ ]:
dataset_cpu = data_samples.cpu().numpy().reshape(-1,n_particles,dimensions)
pool_biased_cpu = target_x.cpu().numpy().reshape(-1,n_particles,dimensions)

fig_size = (20 * 0.393701, 10 * 0.393701)
fig, ax = plt.subplots(1, 2, figsize = fig_size, dpi = 400)

ax[0].set_aspect('equal')
ax[0].scatter(dataset_cpu[:,:,0], dataset_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C0")
ax[0].set_xlim(-box_length/2, box_length/2)
ax[0].set_ylim(-box_length/2, box_length/2)

ax[1].set_aspect('equal')
ax[1].scatter(pool_biased_cpu[:,:,0], pool_biased_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C2")
ax[1].set_xlim(-box_length/2, box_length/2)
ax[1].set_ylim(-box_length/2, box_length/2)

# ax[2].set_aspect('equal')
# ax[2].scatter(pool_cpu[:,:,0], pool_cpu[:,:,1], s=.25, zorder = 10, alpha=0.04, color="C3")
# ax[2].set_xlim(-box_length/2, box_length/2)
# ax[2].set_ylim(-box_length/2, box_length/2)

plt.show()